# Introduction to SageMaker JumpStart - Qwen3.5-27B-FP8 (boto3)

This notebook demonstrates how to deploy and invoke a SageMaker JumpStart model using **boto3** directly,
without the SageMaker Python SDK.

We use the SageMaker `describe_hub_content` API to resolve model artifacts (container image, model data,
environment variables) from the SageMaker Public Hub, then deploy using standard boto3 SageMaker calls.

## Setup

Only `boto3` is required. No SageMaker Python SDK needed.

In [ ]:
%pip install --upgrade boto3

In [1]:
import boto3
import json
import time
from datetime import datetime

## Configuration

In [2]:
model_id = "huggingface-vlm-qwen3-5-27b-fp8"
instance_type = "ml.g6.24xlarge"  # Default instance type for this model

session = boto3.Session()
region = session.region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

endpoint_name = f"qwen3-5-27b-fp8-boto3-{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Instance type: {instance_type}")
print(f"Endpoint name: {endpoint_name}")

Region: us-east-1
Account: 797671033962
Instance type: ml.g6.24xlarge
Endpoint name: qwen3-5-27b-fp8-boto3-20260527152835


## Resolve JumpStart model artifacts from the Public Hub

We use `describe_hub_content` to retrieve the model's metadata document, which contains
the container image URI, model data S3 location, and environment variables needed for deployment.

In [3]:
sm_client = boto3.client("sagemaker", region_name=region)
smr_client = boto3.client("sagemaker-runtime", region_name=region)

# Retrieve model metadata from SageMaker Public Hub
hub_response = sm_client.describe_hub_content(
    HubName="SageMakerPublicHub",
    HubContentType="Model",
    HubContentName=model_id,
)

model_doc = json.loads(hub_response["HubContentDocument"])
print(f"Model: {hub_response['HubContentDisplayName']}")
print(f"Version: {hub_response['HubContentVersion']}")
print(f"Supported instances: {model_doc.get('SupportedInferenceInstanceTypes', [])}")
print(f"Default instance: {model_doc.get('DefaultInferenceInstanceType', 'N/A')}")

Model: Qwen3.5-27B-FP8
Version: 1.0.1
Supported instances: ['ml.g6.24xlarge']
Default instance: ml.g6.24xlarge


In [4]:
# Extract deployment artifacts from the model document
# The hub document uses these top-level keys:
#   HostingEcrUri - container image
#   HostingArtifactUri - model data in S3
#   InferenceEnvironmentVariables - base env vars (list of {Name, Default})
#   HostingInstanceTypeVariants - per-instance-family overrides

image_uri = model_doc["HostingEcrUri"]
model_data_url = model_doc["HostingArtifactUri"]
model_data_type = model_doc.get("HostingArtifactS3DataType", "S3Prefix")
compression_type = model_doc.get("HostingArtifactCompressionType", "None")

# Convert env vars from list of {Name, Default} to flat dict
raw_env = model_doc.get("InferenceEnvironmentVariables", [])
environment = {item["Name"]: str(item["Default"]) for item in raw_env}

# Merge instance-type-specific environment variables
# Check for exact instance type match first, then instance family
instance_family = instance_type.split(".")[1]  # e.g., 'g6' from 'ml.g6.24xlarge'
variants = model_doc.get("HostingInstanceTypeVariants", {}).get("Variants", {})

variant = variants.get(instance_type, variants.get(instance_family, {}))
variant_env = variant.get("Properties", {}).get("EnvironmentVariables", {})
environment.update(variant_env)

# Also check if the variant overrides the image URI
variant_image = variant.get("Properties", {}).get("ImageUri", "")
if variant_image:
    image_uri = variant_image

print(f"Image URI: {image_uri}")
print(f"Model data URL: {model_data_url}")
print(f"Model data type: {model_data_type}")
print(f"Compression: {compression_type}")
print(f"\nEnvironment variables:")
for k, v in environment.items():
    print(f"  {k} = {v}")

Image URI: 763104351884.dkr.ecr.us-east-1.amazonaws.com/vllm:0.17-gpu-py312-cu129-ubuntu22.04-sagemaker-v1
Model data URL: s3://jumpstart-cache-prod-us-east-1/huggingface-vlm/huggingface-vlm-qwen3-5-27b-fp8/artifacts/inference-prepack/v1.0.0/
Model data type: S3Prefix
Compression: None

Environment variables:
  ENDPOINT_SERVER_TIMEOUT = 3600
  HF_MODEL_ID = /opt/ml/model
  MODEL_CACHE_ROOT = /opt/ml/model
  SAGEMAKER_CONTAINER_LOG_LEVEL = 20
  SAGEMAKER_ENV = 1
  SAGEMAKER_MODEL_SERVER_TIMEOUT = 3600
  SAGEMAKER_MODEL_SERVER_WORKERS = 1
  SAGEMAKER_PROGRAM = inference.py
  SAGEMAKER_SUBMIT_DIRECTORY = /opt/ml/model/code
  HF_HUB_OFFLINE = 1
  MAX_BATCH_SIZE = 16
  MAX_CONCURRENT_REQUESTS = 5
  OPTION_TOOL_CALL_PARSER = qwen3_coder
  SM_VLLM_DTYPE = auto
  SM_VLLM_ENABLE_CHUNKED_PREFILL = true
  SM_VLLM_GPU_MEMORY_UTILIZATION = 0.85
  SM_VLLM_MAX_MODEL_LEN = 2275
  SM_VLLM_MAX_NUM_SEQS = 16
  SM_VLLM_MODEL = /opt/ml/model
  SM_VLLM_TENSOR_PARALLEL_SIZE = 4
  TRANSFORMERS_OFFLINE = 1


## Get IAM execution role

If running on SageMaker Studio/Notebook instances, the execution role is available from instance metadata.
Otherwise, set it manually.

In [5]:
import os
import urllib.request

def get_execution_role():
    """Get the SageMaker execution role from notebook instance metadata or environment."""
    # Check if role is set as environment variable
    role = os.environ.get("SAGEMAKER_ROLE_ARN") or os.environ.get("AWS_SAGEMAKER_ROLE")
    if role:
        return role

    # Try to get from SageMaker notebook instance metadata
    try:
        req = urllib.request.Request(
            "http://169.254.169.254/latest/meta-data/iam/security-credentials/"
        )
        with urllib.request.urlopen(req, timeout=2) as resp:
            role_name = resp.read().decode("utf-8").strip()
        iam = boto3.client("iam")
        role_info = iam.get_role(RoleName=role_name)
        return role_info["Role"]["Arn"]
    except Exception:
        pass

    # Try SageMaker Studio metadata
    try:
        if os.path.exists("/opt/ml/metadata/resource-metadata.json"):
            with open("/opt/ml/metadata/resource-metadata.json") as f:
                metadata = json.load(f)
            domain_id = metadata.get("DomainId")
            user_profile = metadata.get("UserProfileName")
            if domain_id and user_profile:
                sm = boto3.client("sagemaker")
                resp = sm.describe_user_profile(
                    DomainId=domain_id, UserProfileName=user_profile
                )
                return resp["UserSettings"]["ExecutionRole"]
    except Exception:
        pass

    raise ValueError(
        "Could not determine SageMaker execution role. "
        "Please set it manually: role_arn = 'arn:aws:iam::ACCOUNT:role/ROLE_NAME'"
    )

role_arn = get_execution_role()
print(f"Using role: {role_arn}")

Using role: arn:aws:iam::797671033962:role/service-role/AmazonSageMaker-ExecutionRole-20260310T192866


In [6]:
# If the above cell fails, set your role ARN manually here:
# role_arn = "<YOUR SAGEMAKER EXECUTION ROLE>"

## Deploy model using boto3

We create a Model, Endpoint Configuration, and Endpoint using the boto3 SageMaker client.

In [7]:
model_name = endpoint_name
endpoint_config_name = endpoint_name

In [8]:
# Step 1: Create Model
print("Creating model...")

primary_container = {
    "Image": image_uri,
    "Environment": environment,
}

# For uncompressed model artifacts (S3Prefix with no compression), use ModelDataSource
if model_data_type == "S3Prefix" and compression_type == "None":
    primary_container["ModelDataSource"] = {
        "S3DataSource": {
            "S3Uri": model_data_url,
            "S3DataType": "S3Prefix",
            "CompressionType": "None",
        }
    }
else:
    primary_container["ModelDataUrl"] = model_data_url

create_model_response = sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer=primary_container,
    ExecutionRoleArn=role_arn,
)
print(f"Model ARN: {create_model_response['ModelArn']}")

Creating model...
Model ARN: arn:aws:sagemaker:us-east-1:797671033962:model/qwen3-5-27b-fp8-boto3-20260527152835


In [9]:
# Step 2: Create Endpoint Configuration
print("Creating endpoint configuration...")

# Use timeouts from the model document (large models need more time)
model_download_timeout = model_doc.get("ModelDataDownloadTimeout", 1200)
container_startup_timeout = model_doc.get("ContainerStartupHealthCheckTimeout", 1200)

production_variant = {
    "VariantName": "AllTraffic",
    "ModelName": model_name,
    "InstanceType": instance_type,
    "InitialInstanceCount": 1,
    "ModelDataDownloadTimeoutInSeconds": model_download_timeout,
    "ContainerStartupHealthCheckTimeoutInSeconds": container_startup_timeout,
}

# VolumeSizeInGB is only supported for instances with EBS-backed storage (e.g., g5, p4d)
# Instances like g6, g6e have local NVMe storage and don't support this parameter
ebs_volume_instance_families = {"g5", "p4d", "p4de", "p5", "p5e", "p5en"}
if instance_family in ebs_volume_instance_families:
    volume_size = model_doc.get("InferenceVolumeSize", 256)
    production_variant["VolumeSizeInGB"] = volume_size

create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[production_variant],
)
print(f"Endpoint Config ARN: {create_endpoint_config_response['EndpointConfigArn']}")

Creating endpoint configuration...
Endpoint Config ARN: arn:aws:sagemaker:us-east-1:797671033962:endpoint-config/qwen3-5-27b-fp8-boto3-20260527152835


In [10]:
# Step 3: Create Endpoint
print("Creating endpoint...")
create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"Endpoint ARN: {create_endpoint_response['EndpointArn']}")

Creating endpoint...
Endpoint ARN: arn:aws:sagemaker:us-east-1:797671033962:endpoint/qwen3-5-27b-fp8-boto3-20260527152835


In [11]:
# Step 4: Wait for endpoint to be InService
print("Waiting for endpoint to be InService (this may take several minutes)...")

waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=endpoint_name,
    WaiterConfig={"Delay": 30, "MaxAttempts": 60}
)

resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
print(f"Endpoint status: {resp['EndpointStatus']}")

Waiting for endpoint to be InService (this may take several minutes)...
Endpoint status: InService


### (Optional) Use an existing endpoint

If you already have a deployed endpoint, set the `endpoint_name` below and skip the deployment cells above.

In [ ]:
# endpoint_name = "<YOUR EXISTING ENDPOINT NAME>"

## Invoke the endpoint

Use the `sagemaker-runtime` boto3 client to send inference requests.

In [12]:
def invoke_endpoint(payload):
    """Invoke the SageMaker endpoint and return the parsed JSON response."""
    response = smr_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read().decode("utf-8"))

In [13]:
# Basic text generation example
payload = {
    "messages": [
        {"role": "user", "content": "What is deep learning?"}
    ],
    "max_tokens": 256,
}

response = invoke_endpoint(payload)
generated_text = response["choices"][0]["message"]["content"]
print("Input: What is deep learning?")
print(f"\nOutput:\n{generated_text.strip()}")

Input: What is deep learning?

Output:
Here's a thinking process that leads to the explanation of deep learning:

1.  **Deconstruct the Request:**
    *   **Question:** "What is deep learning?"
    *   **Intent:** The user wants a clear, comprehensive, yet accessible definition and explanation of deep learning. They might be a beginner, a student, or someone curious about AI trends.
    *   **Key Concepts to Cover:** Definition, relation to AI/ML, neural networks, "deep" meaning, how it works (briefly), applications, pros/cons.

2.  **Initial Brainstorming & Structuring:**
    *   *Analogy:* How do I explain it simply? (Brain/Neurons).
    *   *Hierarchy:* AI > Machine Learning > Deep Learning.
    *   *Core Mechanism:* Artificial Neural Networks (ANNs), layers, weights, backpropagation.
    *   *Why "Deep"?* Many hidden layers.
    *   *Use Cases:* Images, text, speech, games.
    *   *Requirements:* Data, compute power.
    *   *Structure:*
        1.  High-level definition.
        

## Invoke the endpoint with no reasoning

Disable the model's thinking/reasoning mode using `chat_template_kwargs`.

In [14]:
payload = {
    "messages": [
        {"role": "user", "content": "Hi, tell me a joke"}
    ],
    "max_tokens": 128,
    "temperature": 0.6,
    "chat_template_kwargs": {"enable_thinking": False}
}

response = invoke_endpoint(payload)
print(response["choices"][0]["message"]["content"])

Why don't scientists trust atoms?

Because they **make up everything**! 😄


## Invoke the endpoint with a dummy Hebrew CV

Send an image (base64-encoded) along with a text prompt for vision-language inference.

In [15]:
import base64

# Encode your image
with open("dummy_cv_heb_1.png", "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                },
                {
                    "type": "text",
                    "text": "Convert this document into Markdown while preserving the structure"
                }
            ]
        }
    ],
    "max_tokens": 1024,
    "temperature": 0.0,
    "chat_template_kwargs": {"enable_thinking": False}
}

response = invoke_endpoint(payload)
print(response["choices"][0]["message"]["content"])

```markdown
# איתי לוי
## Full Stack מפתח

מפתח תוכנה עם 4+ שנות ניסיון בפיתוח יישומי Web מאפס ועד פרודקשן.
מתמחה ב-React, Node.js, TypeScript, React Native ו-Python.
אוהב לפתור בעיות מורכבות, כותב קוד נקי, עובד היטב בצוות
ומתמיד בלמידה של טכנולוגיות חדשות.

---

## פרטים אישיים

- **אימייל:** itai.levi.dev@gmail.com
- **טלפון:** 054-1234567
- **מיקום:** תל אביב, ישראל
- **לינקדאין:** linkedin.com/in/itai-levi-dev
- **גיטהאב:** github.com/itai-levi

---

## ניסיון תעסוקתי

### מפתח Full Stack
**TechNova**
ינואר 2022 | תל אביב

- פיתוח ותחזוקה של אפליקציות Web בסביבת React, Node.js ו-MongoDB
- עבודה עם טכנולוגיות מודרניות כמו TypeScript ו-Next.js
- שילוב ביצועים ואופטימיזציה של זמני טעינה
- הובלת תהליכי Code Review ונוכחות מפתחים חדשים

### מפתח Full Stack
**DataWave**
יוני 2020 - דצמבר 2021 | רמת גן

- פיתוח פיצ'רים חדשים ושפור מערכות קיימות
- עבודה עם Express.js, PostgreSQL ו-Express.js
- בניית ממשקי משתמש רספונסיביים ב-React ו-Redux
- כתיבת בדיקות אוטומטיות עם Jest ו-Cypress

### מפת

In [16]:
from IPython.display import Markdown, display

display(Markdown(response["choices"][0]["message"]["content"]))

```markdown
# איתי לוי
## Full Stack מפתח

מפתח תוכנה עם 4+ שנות ניסיון בפיתוח יישומי Web מאפס ועד פרודקשן.
מתמחה ב-React, Node.js, TypeScript, React Native ו-Python.
אוהב לפתור בעיות מורכבות, כותב קוד נקי, עובד היטב בצוות
ומתמיד בלמידה של טכנולוגיות חדשות.

---

## פרטים אישיים

- **אימייל:** itai.levi.dev@gmail.com
- **טלפון:** 054-1234567
- **מיקום:** תל אביב, ישראל
- **לינקדאין:** linkedin.com/in/itai-levi-dev
- **גיטהאב:** github.com/itai-levi

---

## ניסיון תעסוקתי

### מפתח Full Stack
**TechNova**
ינואר 2022 | תל אביב

- פיתוח ותחזוקה של אפליקציות Web בסביבת React, Node.js ו-MongoDB
- עבודה עם טכנולוגיות מודרניות כמו TypeScript ו-Next.js
- שילוב ביצועים ואופטימיזציה של זמני טעינה
- הובלת תהליכי Code Review ונוכחות מפתחים חדשים

### מפתח Full Stack
**DataWave**
יוני 2020 - דצמבר 2021 | רמת גן

- פיתוח פיצ'רים חדשים ושפור מערכות קיימות
- עבודה עם Express.js, PostgreSQL ו-Express.js
- בניית ממשקי משתמש רספונסיביים ב-React ו-Redux
- כתיבת בדיקות אוטומטיות עם Jest ו-Cypress

### מפתח תוכנה
**SoftLogic**
אוקטובר 2019 - מאי 2020 | ירושלים

- פיתוח מודולים ב-Python (Django)
- עבודה עם מסדי נתונים וכתיבת שאילתות SQL
- תיקון באגים ושפור תהליכי CI/CD
- שיתוף פעולה עם צוותי QA וטוענים

---

## מימנויות

- JavaScript / TypeScript
- React / Next.js
- Node.js / Express
- Python / Django
- SQL / NoSQL
- HTML / CSS / SASS
- Git / GitHub
- Docker

---

## כלים וטכנולוגיות

- React
- Next.js
- Node.js
- TypeScript
- Python
- Django
- PostgreSQL
- MongoDB
- Redis
- Docker
- Git
- Jest
- Cypress
- AWS
- CI/CD
- GraphQL

---

## פרויקטים אישיים

### DevBlog
פלטפורמה לבלוגים למפתחים עם מערכת נ

## Clean up

Delete the endpoint, endpoint configuration, and model to avoid unnecessary charges.

In [17]:
print("Deleting endpoint...")
sm_client.delete_endpoint(EndpointName=endpoint_name)

print("Deleting endpoint configuration...")
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

print("Deleting model...")
sm_client.delete_model(ModelName=model_name)

print("Cleanup complete.")

Deleting endpoint...
Deleting endpoint configuration...
Deleting model...
Cleanup complete.
